### Why So Emotional? A Multi-Modal Explainer for Voice-Based Emotion Recognition

Overview

This project performs:

* Speech-to-Text (ASR) using Wav2Vec2

* Extraction of Attention Weights from the model for interpretability

* Audio Feature Analysis (pitch and energy)

* Explainability combining audio and transcript insights

* Professional Interactive Visualizations using Plotly with clear styling



#### 📦 1. Install & Import Libraries

In [1]:
# Core imports
import torch
import numpy as np
import librosa
import plotly.graph_objects as go
import plotly.subplots as sp

# Hugging Face Transformers for ASR and model internals
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, Wav2Vec2Model

2025-06-02 16:08:22.036971: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748880502.294663      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748880502.375867      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2: Load audio & preprocess

In [2]:
def load_audio(audio_path: str, target_sr: int = 16000):
    """
    Load an audio file and resample to target sampling rate (default 16kHz).

    Args:
        audio_path (str): Path to audio file (wav or other format).
        target_sr (int): Sampling rate for model compatibility.

    Returns:
        waveform (np.ndarray): 1D audio time series.
        sample_rate (int): Sampling rate of the audio.
    """
    waveform, sr = librosa.load(audio_path, sr=target_sr)
    print(f"Loaded audio '{audio_path}' with {waveform.shape[0]} samples at {sr} Hz")
    return waveform, sr

# Example usage
audio_path = "/kaggle/input/ravdess-emotional-speech-audio/Actor_01/03-01-01-01-01-01-01.wav"
waveform, sample_rate = load_audio(audio_path)

Loaded audio '/kaggle/input/ravdess-emotional-speech-audio/Actor_01/03-01-01-01-01-01-01.wav' with 52853 samples at 16000 Hz


3. Load Pretrained ASR Model & Processor

In [3]:
def load_asr_model(model_name="facebook/wav2vec2-base-960h"):
    """
    Load Wav2Vec2 ASR model and processor.

    Args:
        model_name (str): Hugging Face pretrained model name.

    Returns:
        processor: Wav2Vec2Processor for audio processing.
        model: Wav2Vec2ForCTC ASR model.
    """
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2ForCTC.from_pretrained(model_name)
    model.eval()  # Disable dropout
    return processor, model

processor, asr_model = load_asr_model()

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


4. Perform ASR Inference & Decode Transcript

In [4]:
def transcribe_audio(waveform: np.ndarray, processor, model) -> str:
    """
    Run ASR inference to transcribe audio waveform.

    Args:
        waveform (np.ndarray): 1D audio samples.
        processor: Wav2Vec2Processor.
        model: Wav2Vec2ForCTC model.

    Returns:
        transcript (str): Decoded transcription string.
    """
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(inputs.input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcript = processor.decode(predicted_ids[0])
    return transcript

transcript = transcribe_audio(waveform, processor, asr_model)
print(f"Transcript:\n{transcript}")

Transcript:
KIDS ARE TALKING BY THE DOOR


5. Extract Attention Weights for Interpretability

In [5]:
def extract_attention(waveform: np.ndarray):
    """
    Extract attention weights from Wav2Vec2 base model.

    Args:
        waveform (np.ndarray): 1D audio samples.

    Returns:
        attentions (list of torch.Tensor): Attention matrices from each layer.
    """
    base_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base", output_attentions=True)
    base_model.eval()
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = base_model(inputs.input_values)
    attentions = outputs.attentions
    return attentions

attentions = extract_attention(waveform)
print(f"Extracted attentions from {len(attentions)} layers")

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model is using Wav2Vec2SdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `layer_head_mask` not None. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Extracted attentions from 12 layers


6. Audio Feature Analysis: Pitch and Energy

In [6]:
def analyze_audio_features(waveform: np.ndarray, sr: int):
    """
    Compute mean pitch and energy from audio waveform.

    Args:
        waveform (np.ndarray): 1D audio samples.
        sr (int): Sampling rate.

    Returns:
        mean_pitch (float): Average pitch in Hz.
        mean_energy (float): Average root mean square energy.
    """
    pitches, magnitudes = librosa.piptrack(y=waveform, sr=sr)
    pitch_values = pitches[magnitudes > np.median(magnitudes)]
    mean_pitch = pitch_values.mean() if pitch_values.size > 0 else 0.0

    energy = librosa.feature.rms(y=waveform)[0]
    mean_energy = energy.mean()

    return mean_pitch, mean_energy

mean_pitch, mean_energy = analyze_audio_features(waveform, sample_rate)
print(f"Mean pitch: {mean_pitch:.2f} Hz, Mean energy: {mean_energy:.4f}")

Mean pitch: 1778.66 Hz, Mean energy: 0.0023


7. Generate Explainability Statement

In [7]:
def generate_explanation(transcript: str, mean_pitch: float, mean_energy: float):
    """
    Create a textual explanation combining pitch, energy, and transcript keywords.

    Args:
        transcript (str): ASR decoded text.
        mean_pitch (float): Average pitch.
        mean_energy (float): Average energy.

    Returns:
        explanation (str): Concise explanation string.
    """
    explanation_parts = []

    # Pitch-based description
    if mean_pitch < 100:
        explanation_parts.append("low pitch")
    else:
        explanation_parts.append("high pitch")

    # Energy-based description
    if mean_energy < 0.02:
        explanation_parts.append("low energy")
    else:
        explanation_parts.append("high energy")

    # Keyword detection in transcript
    keywords = ["happy", "sad", "angry", "lonely"]
    detected_keywords = [kw for kw in keywords if kw in transcript.lower()]
    for kw in detected_keywords:
        explanation_parts.append(f"word '{kw}' detected")

    explanation = " and ".join(explanation_parts)
    return explanation

explanation = generate_explanation(transcript, mean_pitch, mean_energy)
print(f"Explanation:\n{explanation}")

Explanation:
high pitch and low energy


8. Professional Visualization with Plotly

In [8]:
def plot_results(waveform, attentions, explanation):
    """
    Create an interactive multi-panel plot for waveform, attention, and explanation.

    Args:
        waveform (np.ndarray): Audio samples.
        attentions (list): Attention matrices from model layers.
        explanation (str): Textual explanation.
    """
    # Prepare figure with 3 rows
    fig = sp.make_subplots(
        rows=3, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.15,
        subplot_titles=("Waveform", "Attention Heatmap (Layer 0, Head 0)", "Explanation")
    )

    # Waveform plot
    fig.add_trace(
        go.Scatter(y=waveform, mode='lines', line=dict(color='royalblue'), name='Waveform'),
        row=1, col=1
    )

    # Attention heatmap (take first layer, first head)
    attn_matrix = attentions[0][0, 0].cpu().numpy()
    fig.add_trace(
        go.Heatmap(
            z=attn_matrix,
            colorscale='Viridis',
            colorbar=dict(title='Attention Weight'),
            name='Attention Heatmap'
        ),
        row=2, col=1
    )

    # Explanation text panel
    fig.add_trace(
        go.Scatter(
            x=[0],
            y=[0],
            text=[explanation],
            mode='text',
            textfont=dict(size=16, color='darkred'),
            showlegend=False
        ),
        row=3, col=1
    )

    # Layout settings
    fig.update_layout(
        height=900,
        width=900,
        title_text="Speech Recognition with Attention & Explanation",
        font=dict(family="Arial", size=14),
        margin=dict(t=80, b=50, l=50, r=50),
        plot_bgcolor='white'
    )

    # Clean axis for explanation panel
    fig.update_xaxes(visible=False, row=3, col=1)
    fig.update_yaxes(visible=False, row=3, col=1)

    fig.show()

plot_results(waveform, attentions, explanation)